# 07 Model Interpretation and Final Submission

Notebook nay fit model cuoi cung, sinh feature importance / SHAP, va canh hang submission theo format thuc te cua repo (`Date`, `Revenue`, `COGS`).

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from joblib import dump
from sklearn.base import clone

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'dataset').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root from the current working directory.')

PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.competition_workflow import (
    build_modeling_dataset,
    build_training_frame,
    ensure_stage_directories,
    evaluate_model_candidates,
    get_model_candidates,
    prepare_submission_template,
    )

dirs = ensure_stage_directories(PROJECT_ROOT)
pd.set_option('display.max_columns', None)

c:\Users\LENOVO\anaconda3\envs\datathon\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
modeling_dataset = build_modeling_dataset(PROJECT_ROOT)
submission_template = prepare_submission_template(PROJECT_ROOT).sort_values('Date').reset_index(drop=True)
candidate_models = get_model_candidates()

revenue_scores = evaluate_model_candidates(modeling_dataset, target_col='Revenue', n_splits=5)
selected_revenue_model_name = revenue_scores.iloc[0]['Model']
selected_cogs_model_name = 'Ridge' if 'Ridge' in candidate_models else selected_revenue_model_name

def fit_target_model(target_col, model_name):
    X, y = build_training_frame(modeling_dataset, target_col=target_col)
    model = clone(candidate_models[model_name])
    model.fit(X, y)
    return model, X.columns.tolist(), X

revenue_model, revenue_feature_columns, revenue_X = fit_target_model('Revenue', selected_revenue_model_name)
cogs_model, cogs_feature_columns, cogs_X = fit_target_model('COGS', selected_cogs_model_name)
revenue_scores

c:\Users\LENOVO\anaconda3\envs\datathon\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=7.95345e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\LENOVO\anaconda3\envs\datathon\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=3.81109e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\LENOVO\anaconda3\envs\datathon\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.22059e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\LENOVO\anaconda3\envs\datathon\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=1.55314e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\LENOVO\

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001773 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3182
[LightGBM] [Info] Number of data points in the train set: 283, number of used features: 44
[LightGBM] [Info] Start training from score 4682440.453401
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

c:\Users\LENOVO\anaconda3\envs\datathon\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=9.8545e-18): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


,Model,MAE,RMSE,R2,Folds
1,XGBoost,735242.330320,1.051396e+06,0.754893,5
2,LightGBM,739215.923189,1.052064e+06,0.754427,5
0,Ridge,793608.426321,1.100910e+06,0.724909,5


In [4]:
exogenous_columns = [
    column for column in revenue_feature_columns
    if not column.startswith('revenue_') and not column.startswith('cogs_')
]
seed_features = modeling_dataset[['date'] + [column for column in exogenous_columns if column in modeling_dataset.columns]].copy()
future_seed = submission_template[['Date']].rename(columns={'Date': 'date'}).merge(seed_features, on='date', how='left').sort_values('date')
future_seed[exogenous_columns] = future_seed[exogenous_columns].ffill().bfill()

def recursive_forecast(target_col, model, feature_columns):
    target_prefix = target_col.lower()
    history = modeling_dataset[['date', target_col]].dropna().sort_values('date').set_index('date')[target_col].to_dict()
    feature_defaults = (
        modeling_dataset.reindex(columns=feature_columns)
        .ffill()
        .bfill()
        .iloc[-1]
        .to_dict()
    )
    predictions = []

    for step, current_date in enumerate(future_seed['date']):
        row = {column: future_seed.loc[future_seed['date'] == current_date, column].iloc[0] for column in exogenous_columns}
        row['dayofweek'] = current_date.dayofweek
        row['month'] = current_date.month
        row['quarter'] = ((current_date.month - 1) // 3) + 1
        row['year'] = current_date.year
        row['dayofmonth'] = current_date.day
        row['weekofyear'] = int(current_date.isocalendar().week)
        row['is_weekend'] = int(current_date.dayofweek >= 5)
        row['is_month_end'] = int(current_date.is_month_end)
        row['time_index'] = len(modeling_dataset) + step

        for lag in [1, 7, 14, 28]:
            lag_date = current_date - pd.Timedelta(days=lag)
            row[f'{target_prefix}_lag_{lag}'] = history.get(lag_date, np.nan)

        for window in [7, 14, 30]:
            values = [history.get(current_date - pd.Timedelta(days=offset), np.nan) for offset in range(1, window + 1)]
            values = [value for value in values if pd.notna(value)]
            row[f'{target_prefix}_roll_mean_{window}'] = float(np.mean(values)) if values else np.nan
            row[f'{target_prefix}_roll_std_{window}'] = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0

        feature_row = pd.DataFrame([{column: row.get(column, np.nan) for column in feature_columns}])
        feature_row = feature_row.fillna(feature_defaults).fillna(0.0)
        prediction = float(model.predict(feature_row)[0])
        history[current_date] = prediction
        predictions.append({'date': current_date, target_col: prediction})

    return pd.DataFrame(predictions)

revenue_forecast = recursive_forecast('Revenue', revenue_model, revenue_feature_columns)
cogs_forecast = recursive_forecast('COGS', cogs_model, cogs_feature_columns)
display(revenue_forecast.head())
display(cogs_forecast.head())

,date,Revenue
0,2023-01-01,1799155.000
1,2023-01-02,1315672.875
2,2023-01-03,1242607.875
3,2023-01-04,1297675.500
4,2023-01-05,1394060.875


,date,COGS
0,2023-01-01,2.156320e+06
1,2023-01-02,2.214796e+06
2,2023-01-03,2.167206e+06
3,2023-01-04,2.178967e+06
4,2023-01-05,2.227437e+06


In [5]:
submission = submission_template.copy()
submission['Revenue'] = revenue_forecast['Revenue'].values
submission['COGS'] = cogs_forecast['COGS'].values
submission.to_csv(PROJECT_ROOT / 'submission.csv', index=False)

feature_importance_values = getattr(revenue_model, 'feature_importances_', np.abs(np.ravel(getattr(revenue_model, 'coef_', np.zeros(len(revenue_feature_columns))))))
feature_importance = pd.DataFrame({
    'feature': revenue_feature_columns,
    'importance': np.ravel(feature_importance_values),
}).sort_values('importance', ascending=False)
feature_importance.to_csv(PROJECT_ROOT / 'feature_importance_final.csv', index=False)
revenue_scores.head(1).to_csv(PROJECT_ROOT / 'validation_final.csv', index=False)
dump({
    'revenue_model_name': selected_revenue_model_name,
    'cogs_model_name': selected_cogs_model_name,
    'revenue_features': revenue_feature_columns,
    'cogs_features': cogs_feature_columns,
    'revenue_model': revenue_model,
    'cogs_model': cogs_model,
}, dirs['models'] / 'final_model.pkl')
submission.head()

,Date,Revenue,COGS
0,2023-01-01,1799155.000,2.156320e+06
1,2023-01-02,1315672.875,2.214796e+06
2,2023-01-03,1242607.875,2.167206e+06
3,2023-01-04,1297675.500,2.178967e+06
4,2023-01-05,1394060.875,2.227437e+06


In [6]:
try:
    background = revenue_X.tail(min(200, len(revenue_X)))
    explainer = shap.Explainer(revenue_model, background)
    shap_values = explainer(background)

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, background, show=False)
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'shap_summary.png', dpi=150, bbox_inches='tight')
    plt.close()

    waterfall_values = shap_values[0]
    plt.figure(figsize=(10, 6))
    shap.plots.waterfall(waterfall_values, show=False)
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'shap_waterfall_sample.png', dpi=150, bbox_inches='tight')
    plt.close()
except Exception as exc:
    print(f'SHAP export skipped: {exc}')